# Derivatives and Partials

**Goal:** Implement scalar and partial derivatives from scratch via central finite differences, validate against `torch.autograd`, and build intuition for how gradients are computed in PyTorch.

## Configuration

Device, random seed, and default dtype come from `shared.config`, which reads `config.toml`.  
On Apple Silicon this resolves to `mps`; on CUDA machines it resolves to `cuda`; otherwise `cpu`.

In [1]:
import sys
from pathlib import Path

import torch


def _find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists():
            return p
    return start


REPO_ROOT = _find_repo_root(Path.cwd())
sys.path.insert(0, str(REPO_ROOT))

from shared.config import configure

device = configure()
print("running on:", device)


running on: mps


## From Scratch: Central Finite Difference (Scalar)

The central finite difference approximation for a scalar derivative is:

```
f'(x) ≈ (f(x + h) - f(x - h)) / (2h)
```

This is second-order accurate in `h`, which avoids the asymmetric truncation error of the one-sided formula.  
We test on `f(x) = x³ + 2x`, whose exact derivative is `f'(x) = 3x² + 2`.

In [2]:
def central_diff(f, x: torch.Tensor, h: float = 1e-5) -> torch.Tensor:
    """Central finite difference derivative for a scalar function f: R -> R.

    Uses float64 on cpu to minimise cancellation error.
    """
    xc = x.detach().cpu().double().reshape(())
    return (f(xc + h) - f(xc - h)) / (2.0 * h)


def f_scalar(x: torch.Tensor) -> torch.Tensor:
    """f(x) = x^3 + 2x.  Exact derivative: 3x^2 + 2."""
    return x ** 3 + 2 * x


# Evaluate at x = 2.0
x0 = torch.tensor(2.0, device=device, requires_grad=True)
fd_deriv = central_diff(f_scalar, x0)
exact_deriv = 3 * 2.0 ** 2 + 2  # = 14.0

print(f"Central diff derivative at x=2: {fd_deriv.item():.8f}")
print(f"Exact derivative at x=2:        {exact_deriv:.8f}")


Central diff derivative at x=2: 14.00000000
Exact derivative at x=2:        14.00000000


## Validation: Finite Differences vs. `torch.autograd.grad`

We compare the finite-difference result against PyTorch's exact derivative via autograd.

In [3]:
# Autograd derivative
x_ag = torch.tensor(2.0, device=device, requires_grad=True)
y = f_scalar(x_ag)
grad_ag, = torch.autograd.grad(y, x_ag)

print(f"Autograd derivative at x=2:     {grad_ag.cpu().item():.8f}")
print(f"Central diff derivative at x=2: {fd_deriv.item():.8f}")

assert torch.allclose(
    torch.tensor(fd_deriv.item()).float(),
    torch.tensor(grad_ag.cpu().item()).float(),
    atol=1e-4,
), "Finite difference does not match autograd!"
print("Assertion passed: finite diff matches autograd ✓")


Autograd derivative at x=2:     14.00000000
Central diff derivative at x=2: 14.00000000
Assertion passed: finite diff matches autograd ✓


## Step Size Sweep: Truncation vs. Cancellation Error

A small `h` reduces truncation error but amplifies floating-point cancellation.  
For `float64` the sweet spot is around `h ≈ 1e-5`.

In [4]:
h_values = [10 ** (-k) for k in range(1, 9)]
# Use cpu float64 to show both regimes clearly
x_cpu = torch.tensor(2.0).double()
exact = 14.0

print(f"{'h':>12s}  {'approx':>16s}  {'abs_err':>12s}")
for h in h_values:
    approx = central_diff(f_scalar, x_cpu, h=h).item()
    err = abs(approx - exact)
    print(f"{h:>12.0e}  {approx:>16.10f}  {err:>12.2e}")


           h            approx       abs_err
       1e-01     14.0100000000      1.00e-02
       1e-02     14.0001000000      1.00e-04
       1e-03     14.0000010000      1.00e-06
       1e-04     14.0000000100      1.00e-08
       1e-05     14.0000000002      1.81e-10
       1e-06     14.0000000002      1.81e-10
       1e-07     13.9999999949      5.15e-09
       1e-08     13.9999999149      8.51e-08


## From Scratch: Partial Derivatives

For a multivariate function `f: R^d -> R`, the partial derivative with respect to coordinate `j` holds all other coordinates fixed and applies the central-difference formula along axis `j`.

We use `f(x, y) = x² + 3xy + y³`.  Exact partials: `∂f/∂x = 2x + 3y`,  `∂f/∂y = 3x + 3y²`.

In [5]:
def partial_diff(f, x: torch.Tensor, j: int, h: float = 1e-5) -> torch.Tensor:
    """Central finite difference partial derivative of f w.r.t. coordinate j.

    Uses float64 on cpu to eliminate float32 cancellation noise for cubic terms.
    """
    x_f = x.detach().cpu().double().clone()
    xph = x_f.clone()
    xph[j] = xph[j] + h
    xmh = x_f.clone()
    xmh[j] = xmh[j] - h
    return (f(xph) - f(xmh)) / (2.0 * h)


def f_multi(v: torch.Tensor) -> torch.Tensor:
    """f(x, y) = x^2 + 3*x*y + y^3."""
    x, y = v[0], v[1]
    return x ** 2 + 3 * x * y + y ** 3


# Evaluate at (x, y) = (1.0, 2.0)
xy = torch.tensor([1.0, 2.0])
df_dx_fd = partial_diff(f_multi, xy, j=0)
df_dy_fd = partial_diff(f_multi, xy, j=1)

# Exact: df/dx = 2(1) + 3(2) = 8; df/dy = 3(1) + 3(2^2) = 15
print(f"partial f/partial x finite diff at (1,2): {df_dx_fd.item():.8f}  (exact: 8.0)")
print(f"partial f/partial y finite diff at (1,2): {df_dy_fd.item():.8f}  (exact: 15.0)")


partial f/partial x finite diff at (1,2): 8.00000000  (exact: 8.0)
partial f/partial y finite diff at (1,2): 15.00000000  (exact: 15.0)


## Validation: Partials via `torch.autograd.grad`

Autograd computes all partial derivatives at once through the gradient vector,
which equals the stacked partial derivatives.

In [6]:
# Autograd gradient of f_multi — on cpu to avoid mps dtype friction
xy_ag = torch.tensor([1.0, 2.0], requires_grad=True)
out = f_multi(xy_ag)
grad_ag, = torch.autograd.grad(out, xy_ag)

print(f"Autograd gradient at (1,2): {grad_ag.tolist()}")
print(f"Expected:                   [8.0, 15.0]")

# The gradient vector equals the stacked partials
fd_grad = torch.tensor([df_dx_fd.item(), df_dy_fd.item()]).float()
assert torch.allclose(fd_grad, grad_ag.float(), atol=1e-4), (
    f"Finite-difference partials do not match autograd! diff={fd_grad - grad_ag}"
)
print("Assertion passed: finite-diff partials match autograd gradient vector ✓")
print(f"\ngrad f = [df/dx, df/dy] = {grad_ag.tolist()} — the gradient vector equals the stacked partials.")


Autograd gradient at (1,2): [8.0, 15.0]
Expected:                   [8.0, 15.0]
Assertion passed: finite-diff partials match autograd gradient vector ✓

grad f = [df/dx, df/dy] = [8.0, 15.0] — the gradient vector equals the stacked partials.


## Idiomatic PyTorch: `tensor.backward()` and `tensor.grad`

In practice, call `loss.backward()` and read `.grad` directly.  
This is the standard pattern used during model training.

In [7]:
# Standard pattern: backward + .grad
xy_std = torch.tensor([1.0, 2.0], device=device, requires_grad=True)
loss = f_multi(xy_std)
loss.backward()

print(f"xy.grad via .backward(): {xy_std.grad.cpu().tolist()}")
print(f"Computation graph node: {loss.grad_fn}")


xy.grad via .backward(): [8.0, 15.0]
Computation graph node: <AddBackward0 object at 0x10a351210>


## Applied Example: Squared-Error Loss

For a linear model `L(w, b) = (w·x + b − y)²`, the partial derivatives are:
```
partial L / partial w_j = 2r x_j     where r = w·x + b − y
partial L / partial b   = 2r
```
These are exactly what gradient descent needs.

In [8]:
d = 4
torch.manual_seed(0)
x_feat = torch.randn(d, device=device)
y_true = torch.tensor(1.0, device=device)

w = torch.randn(d, device=device, requires_grad=True)
b = torch.zeros(1, device=device, requires_grad=True)

pred = w @ x_feat + b
loss_sq = (pred - y_true) ** 2
loss_sq.backward()

r = (pred - y_true).detach()
manual_dw = 2 * r * x_feat
manual_db = 2 * r

print("dL/dw autograd:", w.grad.cpu().tolist())
print("dL/dw manual:  ", manual_dw.cpu().tolist())
print("dL/db autograd:", b.grad.cpu().tolist())
print("dL/db manual:  ", manual_db.cpu().tolist())

assert torch.allclose(w.grad, manual_dw, atol=1e-5), "w grad mismatch!"
assert torch.allclose(b.grad.squeeze(), manual_db.squeeze(), atol=1e-5), "b grad mismatch!"
print("Assertion passed: manual partials match autograd for squared-error loss ✓")


dL/dw autograd: [-0.9031770825386047, -3.7992990016937256, 2.0291085243225098, -3.1270036697387695]
dL/dw manual:   [-0.9031770825386047, -3.7992990016937256, 2.0291085243225098, -3.1270036697387695]
dL/db autograd: [2.645909309387207]
dL/db manual:   [2.645909309387207]
Assertion passed: manual partials match autograd for squared-error loss ✓


## Takeaways

- **Derivative** measures local rate of change: `f'(x) = (f(x+h) - f(x-h)) / (2h)` converges as `h -> 0`.
- **Central differences** are second-order accurate; very small `h` reintroduces floating-point cancellation.
- **Partial derivative** freezes all but one coordinate; the gradient vector stacks all partials.
- **`torch.autograd`** computes exact derivatives via reverse-mode automatic differentiation: it records the computation graph during the forward pass and accumulates gradients by walking it in reverse — not symbolic differentiation, and not finite differences.
- **Finite-difference gradient checks** are the standard tool for validating custom autograd or hand-derived gradients.
- **Squared-error gradients** `partial L/partial w_j = 2r x_j` show that only inputs with large `x_j` and large residual `r` drive large updates.
